# 🗄️ Day 5B — Build the Tax Knowledge Vector Store
**Agentic AI for Tax Technologists · 12-Day Program**

---

## 📖 Where We Are

Notebook 5A showed you what embeddings are and proved that semantically similar tax phrases cluster together.  
Now we store those embeddings in **ChromaDB** — a vector store that indexes them for fast retrieval.

**What you'll build:**
- 20 pre-chunked tax knowledge entries (GST + TDS + corporate tax)
- Each chunk embedded and stored with rich metadata
- 5 semantic queries run against the store
- Side-by-side comparison: RAG retrieval vs keyword matching

---
## ⏱️ Time: 60 minutes

---
## ⚙️ Step 1: Install & Import

In [1]:
%pip install openai chromadb numpy --quiet


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import re
import numpy as np
import chromadb
import datetime
from getpass import getpass
from openai import AzureOpenAI

print("✅ Imports OK")

✅ Imports OK


---
## 🔑 Step 2: Configure Azure OpenAI

In [5]:
AZURE_OPENAI_ENDPOINT       = input("Endpoint (e.g. https://xxx.openai.azure.com/): ").strip()
AZURE_OPENAI_API_KEY        = getpass("API Key: ")
AZURE_DEPLOYMENT_NAME       = input("Chat deployment name (e.g. gpt-4o): ").strip()
AZURE_EMBEDDING_DEPLOYMENT  = input("Embeddings deployment name (e.g. text-embedding-ada-002): ").strip()
AZURE_API_VERSION           = "2024-08-01-preview"

client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
    api_version    = AZURE_API_VERSION,
)
print("✅ Azure OpenAI client initialised successfully!")

Endpoint (e.g. https://xxx.openai.azure.com/):  https://agentic-ai-tt.openai.azure.com/
API Key:  ········
Chat deployment name (e.g. gpt-4o):  gpt-4o
Embeddings deployment name (e.g. text-embedding-ada-002):  text-embedding-ada-002


✅ Azure OpenAI client initialised successfully!


---
## 📚 Step 3: The Tax Knowledge Chunks

In production you'd load PDFs and split them. For this lab we use pre-chunked text so everyone gets
identical results. Each chunk mimics what you'd extract from a real tax circular or notification —
with metadata matching what a section-aware splitter would produce.

In [7]:
TAX_KNOWLEDGE_CHUNKS = [
    # ── GST — Exports ─────────────────────────────────────────────────────────
    {
        "id": "gst_export_001",
        "text": "Export of services is treated as zero-rated supply under Section 16(1) of the IGST Act 2017. "
                "A registered supplier may export without payment of IGST if a Letter of Undertaking (LUT) has been "
                "filed under Rule 96A. Alternatively, the supplier may export on payment of IGST and claim a refund "
                "under Section 54 of the CGST Act. LUT is valid for the entire financial year from the date of filing.",
        "metadata": {"source": "IGST Act 2017", "section": "16(1)", "type": "legislation", "topic": "exports", "year": 2017}
    },
    {
        "id": "gst_export_002",
        "text": "Place of supply for export of services is determined under Section 13 of the IGST Act. "
                "When the recipient is located outside India, the place of supply is the location of the recipient. "
                "IT/ITES services provided to overseas clients qualify as export of services, subject to: "
                "(a) supplier is in India, (b) recipient is outside India, (c) payment received in convertible foreign exchange, "
                "(d) supplier and recipient are not merely establishments of the same person.",
        "metadata": {"source": "IGST Act 2017", "section": "13", "type": "legislation", "topic": "exports", "year": 2017}
    },
    {
        "id": "gst_notification_01_2023",
        "text": "Notification No. 01/2023-Integrated Tax (Rate) clarifies the GST rate schedule for IT and ITES services. "
                "Cloud computing services, SaaS subscriptions, and data processing services supplied by Indian entities "
                "to overseas recipients qualify as export of services and are zero-rated if LUT is filed. "
                "If LUT is not filed, IGST at 18% applies and refund may be claimed. "
                "Domestic B2B supply of these services attracts 18% GST (9% CGST + 9% SGST for intra-state, 18% IGST for inter-state).",
        "metadata": {"source": "Notification 01/2023-IT(Rate)", "section": "4", "type": "notification", "topic": "exports", "year": 2023}
    },
    {
        "id": "gst_composition_001",
        "text": "Section 10 of the CGST Act 2017 provides for the composition scheme. Registered suppliers with aggregate "
                "turnover not exceeding Rs 1.5 crore (Rs 75 lakh for special category states) may opt for composition levy. "
                "Composition dealers pay tax at 1% of turnover for traders, 2% for manufacturers, and 5% for restaurants. "
                "Composition dealers cannot collect tax from recipients, cannot issue tax invoices, "
                "and cannot avail input tax credit. They must issue a Bill of Supply instead.",
        "metadata": {"source": "CGST Act 2017", "section": "10", "type": "legislation", "topic": "composition_scheme", "year": 2017}
    },
    {
        "id": "gst_itc_001",
        "text": "Input Tax Credit (ITC) under Section 16 of the CGST Act can be claimed if: (a) the supplier has filed "
                "returns and paid the tax, (b) the recipient holds a valid tax invoice, (c) the goods or services have "
                "been received, and (d) the tax has been paid to the government. Section 17(5) lists blocked credits "
                "including motor vehicles for personal use, food and beverages, beauty treatment, health services, "
                "and club memberships. ITC cannot be claimed after filing of annual return for the relevant financial year.",
        "metadata": {"source": "CGST Act 2017", "section": "16", "type": "legislation", "topic": "input_tax_credit", "year": 2017}
    },
    {
        "id": "gst_rcm_001",
        "text": "Reverse Charge Mechanism (RCM) under Section 9(3) of the CGST Act requires the recipient to pay GST "
                "instead of the supplier in notified categories. Key RCM notifications include: "
                "legal services by an advocate (Notification 13/2017-CT), services by a director to a company, "
                "goods transport agency (GTA) services, import of services from overseas suppliers, "
                "and security services supplied by an individual to a body corporate. "
                "RCM applies at the standard rate applicable to the service/goods.",
        "metadata": {"source": "CGST Act 2017", "section": "9(3)", "type": "legislation", "topic": "reverse_charge", "year": 2017}
    },
    {
        "id": "gst_registration_001",
        "text": "GST registration is mandatory when aggregate turnover exceeds Rs 20 lakh for service providers "
                "(Rs 10 lakh in special category states) and Rs 40 lakh for goods suppliers. "
                "Inter-state suppliers must register regardless of turnover. E-commerce operators and persons "
                "supplying through e-commerce operators must also register regardless of turnover. "
                "Voluntary registration is permitted below the threshold.",
        "metadata": {"source": "CGST Act 2017", "section": "22", "type": "legislation", "topic": "registration", "year": 2017}
    },
    # ── GST — Filing ──────────────────────────────────────────────────────────
    {
        "id": "gst_gstr1_001",
        "text": "GSTR-1 is the return for outward supplies filed by registered taxpayers. Monthly filers with turnover "
                "above Rs 5 crore must file by the 11th of the following month. Quarterly filers under QRMP scheme "
                "file by the 13th of the month following each quarter. GSTR-1 includes B2B invoices, B2C summary, "
                "exports, debit/credit notes, and HSN summary. Late filing attracts Rs 50/day (Rs 20/day for nil returns).",
        "metadata": {"source": "CGST Rules 2017", "section": "Rule 59", "type": "rule", "topic": "filing", "year": 2017}
    },
    {
        "id": "gst_gstr3b_001",
        "text": "GSTR-3B is the monthly summary return for payment of GST liability. Monthly taxpayers file by the 20th "
                "of the following month. Under QRMP scheme, GSTR-3B is filed quarterly — by the 22nd for Category-I states "
                "and 24th for Category-II states. GSTR-3B covers outward supplies, inward supplies liable to RCM, "
                "eligible ITC, and net tax payable. Payment must accompany filing.",
        "metadata": {"source": "CGST Rules 2017", "section": "Rule 61", "type": "rule", "topic": "filing", "year": 2017}
    },
    # ── TDS ───────────────────────────────────────────────────────────────────
    {
        "id": "tds_194c_001",
        "text": "Section 194C of the Income Tax Act requires TDS deduction on payments to contractors and sub-contractors. "
                "TDS rate is 1% for payments to individuals and HUFs, and 2% for companies and firms. "
                "TDS is deducted when a single payment exceeds Rs 30,000 or aggregate payments in a financial year "
                "exceed Rs 1,00,000. Section 194C also covers advertising contracts, catering, broadcasting, and "
                "carriage of goods. Sub-contractors of the primary contractor are also covered.",
        "metadata": {"source": "Income Tax Act 1961", "section": "194C", "type": "legislation", "topic": "tds", "year": 1961}
    },
    {
        "id": "tds_194j_001",
        "text": "Section 194J covers TDS on fees for professional or technical services. TDS rate is 10% for professional "
                "services (lawyers, doctors, engineers, architects, chartered accountants) and 2% for technical services. "
                "The threshold for deduction is Rs 30,000 per annum per payee. "
                "Professional services include medical, legal, engineering, and management consultancy. "
                "Technical services include managerial, technical, and consultancy services not covered under 194C.",
        "metadata": {"source": "Income Tax Act 1961", "section": "194J", "type": "legislation", "topic": "tds", "year": 1961}
    },
    {
        "id": "tds_return_001",
        "text": "TDS returns must be filed quarterly. Form 24Q covers TDS on salaries, Form 26Q covers TDS on non-salary "
                "payments to residents, and Form 27Q covers TDS on payments to non-residents. Due dates: "
                "Q1 (Apr-Jun): July 31; Q2 (Jul-Sep): October 31; Q3 (Oct-Dec): January 31; Q4 (Jan-Mar): May 31. "
                "TDS certificate in Form 16 (salary) and Form 16A (non-salary) must be issued within 15 days of the "
                "due date of filing the quarterly TDS return.",
        "metadata": {"source": "Income Tax Rules 1962", "section": "Rule 31A", "type": "rule", "topic": "tds", "year": 1962}
    },
    # ── Corporate Tax ─────────────────────────────────────────────────────────
    {
        "id": "corp_tax_rates_001",
        "text": "Corporate income tax rates for Indian domestic companies: 25% for companies with total turnover or "
                "gross receipts not exceeding Rs 400 crore in the preceding year; 30% for all other domestic companies. "
                "Section 115BAB provides for a reduced rate of 15% for new domestic manufacturing companies incorporated "
                "after October 1, 2019. The effective rate including surcharge and health and education cess "
                "(4%) is approximately 25.17% for companies under the 25% slab.",
        "metadata": {"source": "Income Tax Act 1961", "section": "115BAB", "type": "legislation", "topic": "corporate_tax", "year": 2019}
    },
    {
        "id": "transfer_pricing_001",
        "text": "Transfer pricing provisions under Sections 92 to 92F of the Income Tax Act require that international "
                "transactions between associated enterprises be at arm's length price. Methods for determining ALP include: "
                "comparable uncontrolled price (CUP), resale price method, cost plus method, transactional net margin "
                "method (TNMM), and profit split method. Taxpayers with international transactions exceeding Rs 1 crore "
                "must obtain a Transfer Pricing Report in Form 3CEB from a CA. Due date: October 31 of the assessment year.",
        "metadata": {"source": "Income Tax Act 1961", "section": "92-92F", "type": "legislation", "topic": "transfer_pricing", "year": 1961}
    },
    {
        "id": "advance_tax_001",
        "text": "Advance tax is payable in four instalments when estimated tax liability exceeds Rs 10,000 in a financial year. "
                "Instalments: 15% by June 15, 45% cumulative by September 15, 75% cumulative by December 15, "
                "100% by March 15. Interest under Section 234B applies at 1% per month for default in payment of advance tax. "
                "Interest under Section 234C applies for deferment of individual instalments. "
                "Senior citizens with no business income are exempt from advance tax.",
        "metadata": {"source": "Income Tax Act 1961", "section": "208-219", "type": "legislation", "topic": "advance_tax", "year": 1961}
    },
    {
        "id": "section_80c_001",
        "text": "Section 80C of the Income Tax Act allows deductions up to Rs 1,50,000 per year from gross total income. "
                "Eligible investments and payments: life insurance premiums, PPF contributions, ELSS mutual funds, "
                "NSC, 5-year tax-saving fixed deposits, tuition fees for up to two children, EPF contributions, "
                "and principal repayment of home loans. This deduction is available only under the old tax regime. "
                "Under the new tax regime (Section 115BAC), Section 80C deductions are not available.",
        "metadata": {"source": "Income Tax Act 1961", "section": "80C", "type": "legislation", "topic": "deductions", "year": 1961}
    },
    {
        "id": "new_tax_regime_001",
        "text": "New tax regime under Section 115BAC (FY 2025-26 slabs): Nil for income up to Rs 3 lakh, "
                "5% for Rs 3–7 lakh, 10% for Rs 7–10 lakh, 15% for Rs 10–12 lakh, 20% for Rs 12–15 lakh, "
                "30% above Rs 15 lakh. Standard deduction of Rs 75,000 available. Rebate under Section 87A: "
                "zero tax for total income up to Rs 7 lakh. No Chapter VI-A deductions (80C, 80D etc.) or HRA "
                "exemption available. The new regime is the default from AY 2024-25.",
        "metadata": {"source": "Income Tax Act 1961", "section": "115BAC", "type": "legislation", "topic": "income_tax_slabs", "year": 2023}
    },
    {
        "id": "ltcg_001",
        "text": "Long-term capital gains (LTCG) on listed equity shares and equity-oriented mutual funds exceeding "
                "Rs 1 lakh in a financial year are taxed at 10% without indexation benefit under Section 112A. "
                "Holding period for equity: more than 12 months. For unlisted shares and immovable property: "
                "more than 24 months. For debt mutual funds: more than 36 months. "
                "Short-term capital gains on listed equity are taxed at 15% under Section 111A. "
                "The Rs 1 lakh exemption is per assessee per financial year and cannot be carried forward.",
        "metadata": {"source": "Income Tax Act 1961", "section": "112A", "type": "legislation", "topic": "capital_gains", "year": 1961}
    },
    {
        "id": "itr_filing_001",
        "text": "Income tax return filing due dates: July 31 for individuals, HUFs, and non-audit cases; "
                "October 31 for taxpayers subject to tax audit under Section 44AB and all companies; "
                "November 30 for transfer pricing cases. Belated return can be filed up to December 31 "
                "of the assessment year with late fee under Section 234F (Rs 5,000 for income above Rs 5 lakh, "
                "Rs 1,000 for income below Rs 5 lakh). Revised return can be filed up to December 31.",
        "metadata": {"source": "Income Tax Act 1961", "section": "139", "type": "legislation", "topic": "filing", "year": 1961}
    },
    {
        "id": "gst_works_contract_001",
        "text": "Works contract under GST is treated as a supply of service. GST rate for works contracts: "
                "18% for general works contracts; 12% for construction of affordable housing projects, "
                "government-awarded civil structures, and roads. Pure labour contracts attract 18% GST. "
                "Input tax credit on goods and services used in works contracts is allowed to the contractor "
                "against output liability, subject to Section 17(5)(c) restrictions on immovable property construction.",
        "metadata": {"source": "Notification 11/2017-CT(Rate)", "section": "Entry 3", "type": "notification", "topic": "works_contract", "year": 2017}
    },
]

# Fix the stray \n in chunk 5 (formatting artifact)
for chunk in TAX_KNOWLEDGE_CHUNKS:
    chunk["text"] = chunk["text"].replace('{"\n\"', '{"')

print(f"✅ Tax knowledge base: {len(TAX_KNOWLEDGE_CHUNKS)} chunks loaded")
topics = set(c["metadata"]["topic"] for c in TAX_KNOWLEDGE_CHUNKS)
print(f"   Topics covered: {sorted(topics)}")

✅ Tax knowledge base: 20 chunks loaded
   Topics covered: ['advance_tax', 'capital_gains', 'composition_scheme', 'corporate_tax', 'deductions', 'exports', 'filing', 'income_tax_slabs', 'input_tax_credit', 'registration', 'reverse_charge', 'tds', 'transfer_pricing', 'works_contract']


---
## 🔢 Step 4: Embed All Chunks

In [8]:
def get_embedding(text):
    """Get embedding vector from Azure OpenAI."""
    response = client.embeddings.create(
        model = AZURE_EMBEDDING_DEPLOYMENT,
        input = text
    )
    return response.data[0].embedding


print(f"Embedding {len(TAX_KNOWLEDGE_CHUNKS)} chunks...")

all_texts      = [c["text"]     for c in TAX_KNOWLEDGE_CHUNKS]
all_ids        = [c["id"]       for c in TAX_KNOWLEDGE_CHUNKS]
all_metadatas  = [c["metadata"] for c in TAX_KNOWLEDGE_CHUNKS]

# Embed one at a time so we can track progress
all_embeddings = []
for i, text in enumerate(all_texts, 1):
    vec = get_embedding(text)
    all_embeddings.append(vec)
    print(f"  [{i:02d}/{len(all_texts)}] {all_ids[i-1]}", end="\r")

print(f"\n✅ Embedded {len(all_embeddings)} chunks")
print(f"   Vector dimension: {len(all_embeddings[0])}")

Embedding 20 chunks...
  [20/20] gst_works_contract_00123
✅ Embedded 20 chunks
   Vector dimension: 1536


---
## 🗄️ Step 5: Store in ChromaDB

In [9]:
# Initialise ChromaDB in-memory (no server, no files)
chroma_client = chromadb.Client()

# Create a collection (equivalent to a table in a relational DB)
# cosine distance is the right metric for text embeddings
tax_collection = chroma_client.create_collection(
    name     = "tax_knowledge",
    metadata = {"hnsw:space": "cosine"}
)

# Convert metadata values to strings (ChromaDB requires string/int/float/bool)
clean_metadatas = [
    {k: str(v) for k, v in m.items()}
    for m in all_metadatas
]

# Add all chunks in one call
tax_collection.add(
    ids        = all_ids,
    documents  = all_texts,
    embeddings = all_embeddings,
    metadatas  = clean_metadatas,
)

print(f"✅ ChromaDB collection 'tax_knowledge' created")
print(f"   Documents stored: {tax_collection.count()}")
print()
print("ChromaDB is now indexing these vectors for fast approximate nearest-neighbour search.")
print("The index uses HNSW (Hierarchical Navigable Small World) algorithm.")

✅ ChromaDB collection 'tax_knowledge' created
   Documents stored: 20

ChromaDB is now indexing these vectors for fast approximate nearest-neighbour search.
The index uses HNSW (Hierarchical Navigable Small World) algorithm.


---
## 🔍 Step 6: Run Semantic Queries

In [10]:
def retrieve(query, n_results=3, filter_metadata=None):
    """
    Semantic retrieval from the vector store.
    Returns a list of {text, metadata, distance} dicts.
    """
    query_vec = get_embedding(query)
    where = filter_metadata if filter_metadata else None
    results = tax_collection.query(
        query_embeddings = [query_vec],
        n_results        = n_results,
        where            = where,
    )
    chunks = []
    for i in range(len(results["ids"][0])):
        chunks.append({
            "id":       results["ids"][0][i],
            "text":     results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": round(results["distances"][0][i], 4),
        })
    return chunks


def print_retrieval(query, chunks):
    """Pretty-print retrieval results."""
    print(f"\n{'='*70}")
    print(f"  QUERY: {query}")
    print(f"{'='*70}")
    for i, chunk in enumerate(chunks, 1):
        meta = chunk["metadata"]
        sim  = 1 - chunk["distance"]  # ChromaDB returns distance; convert to similarity
        bar  = "█" * int(sim * 20)
        print(f"\n  Rank {i}  [sim={sim:.3f}] {bar}")
        print(f"  Source: {meta.get('source','?')}  §{meta.get('section','?')}  [{meta.get('topic','?')}]")
        print(f"  {chunk['text'][:180]}...")


TEST_QUERIES = [
    "What is the GST rate on cloud software services exported to a US client?",
    "When must I deduct TDS and at what rate for a law firm?",
    "What are the income tax slabs for FY 2025-26?",
    "When is the GST return filing deadline for a monthly taxpayer?",
    "Can a manufacturing company claim ITC on legal services purchased?",
]

for q in TEST_QUERIES:
    chunks = retrieve(q, n_results=3)
    print_retrieval(q, chunks)


  QUERY: What is the GST rate on cloud software services exported to a US client?

  Rank 1  [sim=0.882] █████████████████
  Source: Notification 01/2023-IT(Rate)  §4  [exports]
  Notification No. 01/2023-Integrated Tax (Rate) clarifies the GST rate schedule for IT and ITES services. Cloud computing services, SaaS subscriptions, and data processing services ...

  Rank 2  [sim=0.824] ████████████████
  Source: IGST Act 2017  §16(1)  [exports]
  Export of services is treated as zero-rated supply under Section 16(1) of the IGST Act 2017. A registered supplier may export without payment of IGST if a Letter of Undertaking (LU...

  Rank 3  [sim=0.821] ████████████████
  Source: Notification 11/2017-CT(Rate)  §Entry 3  [works_contract]
  Works contract under GST is treated as a supply of service. GST rate for works contracts: 18% for general works contracts; 12% for construction of affordable housing projects, gove...

  QUERY: When must I deduct TDS and at what rate for a law firm?

  Ran

---
## 🆚 Step 7: RAG Retrieval vs Keyword Search

In [11]:
def keyword_search(query, chunks_list, top_n=3):
    """Simple keyword overlap search — no embeddings."""
    query_words = set(re.findall(r'\b\w{4,}\b', query.lower()))
    scored = []
    for chunk in chunks_list:
        text_words = set(re.findall(r'\b\w{4,}\b', chunk["text"].lower()))
        score = len(query_words & text_words) / max(len(query_words), 1)
        scored.append((score, chunk["id"], chunk["text"][:120]))
    scored.sort(reverse=True)
    return scored[:top_n]


# Comparison on a semantically tricky query
# 'zero-rated levy' won't keyword-match documents saying 'zero-rated supply'
TRICKY_QUERY = "Is the levy on export of digital consultancy zero-rated?"

print("COMPARISON: RAG vs Keyword Search")
print(f"Query: {TRICKY_QUERY}")
print()

print("RAG (embedding similarity):")
rag_results = retrieve(TRICKY_QUERY, n_results=3)
for i, r in enumerate(rag_results, 1):
    sim = 1 - r["distance"]
    print(f"  {i}. [{sim:.3f}] {r['id']:<30} {r['text'][:90]}...")

print()
print("Keyword search:")
kw_results = keyword_search(TRICKY_QUERY, TAX_KNOWLEDGE_CHUNKS, top_n=3)
for i, (score, chunk_id, snippet) in enumerate(kw_results, 1):
    print(f"  {i}. [{score:.3f}] {chunk_id:<30} {snippet}...")

print()
print("Observation: RAG finds the export + zero-rated chunks even though the query")
print("uses 'levy' and 'digital consultancy' — different words, same meaning.")
print("Keyword search may miss these because the exact words don't appear.")

COMPARISON: RAG vs Keyword Search
Query: Is the levy on export of digital consultancy zero-rated?

RAG (embedding similarity):
  1. [0.833] gst_export_001                 Export of services is treated as zero-rated supply under Section 16(1) of the IGST Act 201...
  2. [0.831] gst_notification_01_2023       Notification No. 01/2023-Integrated Tax (Rate) clarifies the GST rate schedule for IT and ...
  3. [0.808] new_tax_regime_001             New tax regime under Section 115BAC (FY 2025-26 slabs): Nil for income up to Rs 3 lakh, 5%...

Keyword search:
  1. [0.500] gst_notification_01_2023       Notification No. 01/2023-Integrated Tax (Rate) clarifies the GST rate schedule for IT and ITES services. Cloud computing...
  2. [0.500] gst_export_001                 Export of services is treated as zero-rated supply under Section 16(1) of the IGST Act 2017. A registered supplier may e...
  3. [0.167] tds_194j_001                   Section 194J covers TDS on fees for professional or technical 

---
## 💾 Step 8: Inspect a Retrieved Chunk in Full

In [12]:
# Get the top result for the export query and inspect it fully
query = "What GST rules apply to export of IT services with LUT?"
results = retrieve(query, n_results=1)
top = results[0]

print("TOP RETRIEVED CHUNK — FULL CONTENT")
print("-" * 60)
print(f"Chunk ID   : {top['id']}")
print(f"Similarity : {1 - top['distance']:.3f}")
print(f"Source     : {top['metadata']['source']}")
print(f"Section    : {top['metadata']['section']}")
print(f"Topic      : {top['metadata']['topic']}")
print()
print("Full text:")
print(top['text'])
print()
print("This is exactly what gets injected into the prompt in Notebook 5C.")
print("The LLM generates its answer from THIS content — not from training memory.")

TOP RETRIEVED CHUNK — FULL CONTENT
------------------------------------------------------------
Chunk ID   : gst_notification_01_2023
Similarity : 0.894
Source     : Notification 01/2023-IT(Rate)
Section    : 4
Topic      : exports

Full text:
Notification No. 01/2023-Integrated Tax (Rate) clarifies the GST rate schedule for IT and ITES services. Cloud computing services, SaaS subscriptions, and data processing services supplied by Indian entities to overseas recipients qualify as export of services and are zero-rated if LUT is filed. If LUT is not filed, IGST at 18% applies and refund may be claimed. Domestic B2B supply of these services attracts 18% GST (9% CGST + 9% SGST for intra-state, 18% IGST for inter-state).

This is exactly what gets injected into the prompt in Notebook 5C.
The LLM generates its answer from THIS content — not from training memory.


---
## 🎓 Key Insight

```
ChromaDB stores:  vector + document text + metadata
Query time:       embed query → find k nearest vectors → return documents
Result:           3 highly relevant chunks, ready to inject into a prompt
```

The vector store does the hard work — similarity search over 20 (or 20,000) documents in milliseconds.

---
## ➡️ Next: Notebook 5C — RAG Query Pipeline

We wire this retrieval into Azure OpenAI generation:
`question → embed → retrieve → augment prompt → generate cited answer`